In [1]:
import kagglehub
import os
import pandas as pd
import numpy as np


In [2]:
path = kagglehub.dataset_download('sahilchambyal/clean-weather')


100%|██████████| 80.5k/80.5k [00:00<00:00, 536kB/s]

Extracting files...


In [3]:
for dir, _, files in os.walk(path):
    for file in files:
        print(os.path.join(dir, file))

/root/.cache/kagglehub/datasets/sahilchambyal/clean-weather/versions/1/clean_weather.csv


In [4]:
data = pd.read_csv(os.path.join(path, '/root/.cache/kagglehub/datasets/sahilchambyal/clean-weather/versions/1/clean_weather.csv'))

In [27]:
data.head(5)

,Unnamed: 0,tmax,tmin,rain,tmax_tomorrow
0,1970-01-01,60.0,35.0,0.0,52.0
1,1970-01-02,52.0,39.0,0.0,52.0
2,1970-01-03,52.0,35.0,0.0,53.0
3,1970-01-04,53.0,36.0,0.0,52.0
4,1970-01-05,52.0,35.0,0.0,50.0


### Dry Run for testing out theory

In [5]:
np.random.seed(0)
i_weight = np.random.rand(1,2) # shape of 1 row, 2 col
h_weight = np.random.rand(2,2)
o_weight = np.random.rand(2,1)

temps = data['tmax'].tail(3).to_numpy()
temps

array([66., 70., 62.])

In [6]:
x0 = temps[0].reshape(1,1)
x1 = temps[1].reshape(1,1)
x2 = temps[2].reshape(1,1)

In [7]:
xi_0 = x0 @ i_weight

xi_0

array([[36.22169126, 47.20249818]])

In [8]:
xh_0 = np.maximum(xi_0, 0)

xh_0

array([[36.22169126, 47.20249818]])

In [9]:
xo_0 = xh_0 @ o_weight

xo_0

array([[57.94406231]])

In [10]:
xi_1 = x1 @ i_weight

xi_1

array([[38.41694527, 50.06325565]])

In [11]:
xh = xh_0 @ h_weight
xh_1 = np.maximum(0, xh + xi_1)
xh_1

array([[ 80.24761908, 100.28766177]])

In [12]:
xo_1 = xh_1 @ o_weight
xo_1

array([[124.54916092]])

In [13]:
xi_2 = x2 @ i_weight
xh_2 = xh_1 @ h_weight
xh_2 = np.maximum(0, xh_2 + xi_2)
xo_2 = xh_2 @ o_weight
xo_2

array([[190.94853131]])

#### Full Forward Pass

In [14]:
np.random.seed(0)

i_weight = np.random.rand(1, 5) / 5 - .1

h_weight = np.random.rand(5, 5) / 5 - .1
h_bias = np.random.rand(1, 5) / 5 - .1

o_weight = np.random.rand(5, 1) * 50
o_bias = np.random.rand(1, 1)

In [15]:
output = np.zeros(3)
hidden = np.zeros((3,5))
prev_hidden = None
sequence = data['tmax'].tail(3).to_numpy()

for i in range(len(sequence)):
  x = sequence[i].reshape(1,1)
  xi = x @ i_weight
  if prev_hidden is None:
    xh = np.tanh(xi)
  else:
    xh = np.tanh(xi + prev_hidden @ h_weight + h_bias)
  prev_hidden = xh
  hidden[i] = xh

  x0 = xh @ o_weight + o_bias
  output[i] = x0

<ipython-input-15-08f8bb41d794>:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  output[i] = x0


In [16]:
output

array([74.31470595, 80.66149404, 77.67852446])

In [17]:
hidden

array([[ 0.56784618,  0.99320288,  0.87557333,  0.53166114, -0.76483255],
       [ 0.58366756,  0.99568651,  0.90034879,  0.69338529, -0.84149203],
       [ 0.5383306 ,  0.99164251,  0.86287584,  0.66091071, -0.80543591]])

#### FUll Backward Pass

In [18]:
def mse(actual, predicted):
  return np.mean(np.square(actual - predicted))
def mse_grad(actual, predicted):
  return (predicted - actual)

In [19]:
actual = np.array([70, 62, 65])

loss_grad = mse_grad(actual, output)
loss_grad

array([ 4.31470595, 18.66149404, 12.67852446])

In [22]:
next_hidden = None
o_weight_grad = o_bias_grad = h_weight_grad = h_bias_grad = i_weight_grad = [0]

for i in range(2, -1, -1):
  l_grad = loss_grad[i].reshape(1, 1)

  # o_weight_grad += hidden[i].T @ l_grad
  o_weight_grad += hidden[i][:, np.newaxis] @ l_grad

  o_bias_grad += np.mean(l_grad)

  o_grad = l_grad @ o_weight.T

  if next_hidden is None:
    h_grad = o_grad
  else:
    h_grad = o_grad + next_hidden @ h_weight.T

  tanh_deriv = 1 - hidden[i,:][np.newaxis,:]
  h_grad = np.multiply(h_grad, tanh_deriv)

  next_hidden = h_grad

  if i > 0:
    h_weight_grad += hidden[i-1][:, np.newaxis] @ h_grad
    h_bias_grad += np.mean(h_grad)

  i_weight_grad += sequence[i].reshape(1,1).T @ h_grad

In [24]:
i_weight_grad

array([[ 32352.67091083,    392.99655233,   9528.43418739,
         35514.29142052, 146401.10347147]])

In [25]:
lr = 1e-6
i_weight -= i_weight_grad * lr
h_weight -= h_weight_grad * lr
h_bias -= h_bias_grad * lr
o_weight -= o_weight_grad * lr
o_bias -= o_bias_grad * lr

In [26]:
i_weight

array([[-0.02258997,  0.04264488,  0.01102424, -0.02653765, -0.16167014]])

### Complete Implementation of final model


In [52]:
from sklearn.preprocessing import StandardScaler
import math

# Define predictors and target
PREDICTORS = ["tmax", "tmin", "rain"]
TARGET = "tmax_tomorrow"

# Scale our data to have mean 0
scaler = StandardScaler()
data[PREDICTORS] = scaler.fit_transform(data[PREDICTORS])

# Split into train, valid, test sets
np.random.seed(0)
split_data = np.split(data, [int(.7*len(data)), int(.85*len(data))])
(train_x, train_y), (valid_x, valid_y), (test_x, test_y) = [[d[PREDICTORS].to_numpy(), d[[TARGET]].to_numpy()] for d in split_data]

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [42]:
# from sklearn.preprocessing import StandardScaler

# scaler_x = StandardScaler()
# scaler_y = StandardScaler()

# train_x = scaler_x.fit_transform(train_x)
# train_y = scaler_y.fit_transform(train_y.reshape(-1, 1)).flatten()

# valid_x = scaler_x.transform(valid_x)
# valid_y = scaler_y.transform(valid_y.reshape(-1, 1)).flatten()


In [53]:
def init_params(layer_conf):
    layers = []
    for i in range(1, len(layer_conf)):
        np.random.seed(0)
        k = 1/math.sqrt(layer_conf[i]["hidden"])
        i_weight = np.random.rand(layer_conf[i-1]["units"], layer_conf[i]["hidden"]) * 2 * k - k

        h_weight = np.random.rand(layer_conf[i]["hidden"], layer_conf[i]["hidden"]) * 2 * k - k
        h_bias = np.random.rand(1, layer_conf[i]["hidden"]) * 2 * k - k

        o_weight = np.random.rand(layer_conf[i]["hidden"], layer_conf[i]["output"]) * 2 * k - k
        o_bias = np.random.rand(1, layer_conf[i]["output"]) * 2 * k - k

        layers.append(
            [i_weight, h_weight, h_bias, o_weight, o_bias]
        )
    return layers

In [46]:
# layer_conf = [
#     {'type': 'input', 'units': 3},
#     {'type': 'rnn', 'hidden': 4, 'output': 1}
# ]

In [54]:
def forward(x, layers):
    hiddens = []
    outputs = []
    for i in range(len(layers)):
        i_weight, h_weight, h_bias, o_weight, o_bias = layers[i]
        hidden = np.zeros((x.shape[0], i_weight.shape[1]))
        output = np.zeros((x.shape[0], o_weight.shape[1]))
        for j in range(x.shape[0]):
            input_x = x[j,:][np.newaxis,:] @ i_weight
            hidden_x = input_x + hidden[max(j-1,0),:][np.newaxis,:] @ h_weight + h_bias
            # Activation.  tanh avoids outputs getting larger and larger.
            hidden_x = np.tanh(hidden_x)
            # Store hidden for use in backprop
            hidden[j,:] = hidden_x

            # Output layer
            output_x = hidden_x @ o_weight + o_bias
            output[j,:] = output_x
        hiddens.append(hidden)
        outputs.append(output)
    return hiddens, outputs[-1]

In [55]:
def backward(layers, x, lr, grad, hiddens):
    for i in range(len(layers)):
        i_weight, h_weight, h_bias, o_weight, o_bias = layers[i]
        hidden = hiddens[i]
        next_h_grad = None
        i_weight_grad, h_weight_grad, h_bias_grad, o_weight_grad, o_bias_grad = [0] * 5

        for j in range(x.shape[0] - 1, -1, -1):
            # Add newaxis in the first dimension
            out_grad = grad[j,:][np.newaxis, :]

            # Output updates
            # np.newaxis creates a size 1 axis, in this case transposing matrix
            o_weight_grad += hidden[j,:][:, np.newaxis] @ out_grad
            o_bias_grad += out_grad

            # Propagate gradient to hidden unit
            h_grad = out_grad @ o_weight.T

            if j < x.shape[0] - 1:
                # Then we multiply the gradient by the hidden weights to pull gradient from next hidden state to current hidden state
                hh_grad = next_h_grad @ h_weight.T
                # Add the gradients together to combine output contribution and hidden contribution
                h_grad += hh_grad

            # Pull the gradient across the current hidden nonlinearity
            # derivative of tanh is 1 - tanh(x) ** 2
            # So we take the output of tanh (next hidden state), and plug in
            tanh_deriv = 1 - hidden[j][np.newaxis,:] ** 2

            # next_h_grad @ np.diag(tanh_deriv_next) multiplies each element of next_h_grad by the deriv
            # Effect is to pull value across nonlinearity
            h_grad = np.multiply(h_grad, tanh_deriv)

            # Store to compute h grad for previous sequence position
            next_h_grad = h_grad.copy()

            # If we're not at the very beginning
            if j > 0:
                # Multiply input from previous layer by post-nonlinearity grad at current layer
                h_weight_grad += hidden[j-1][:, np.newaxis] @ h_grad
                h_bias_grad += h_grad

            i_weight_grad += x[j,:][:,np.newaxis] @ h_grad

        # Normalize lr by number of sequence elements
        lr = lr / x.shape[0]
        i_weight -= i_weight_grad * lr
        h_weight -= h_weight_grad * lr
        h_bias -= h_bias_grad * lr
        o_weight -= o_weight_grad * lr
        o_bias -= o_bias_grad * lr
        layers[i] = [i_weight, h_weight, h_bias, o_weight, o_bias]
    return layers

In [ ]:
epochs = 250
lr = 1e-5

layer_conf = [
    {"type":"input", "units": 3},
    {"type": "rnn", "hidden": 4, "output": 1}
]
layers = init_params(layer_conf)

for epoch in range(epochs):
    sequence_len = 7
    epoch_loss = 0
    for j in range(train_x.shape[0] - sequence_len):
        seq_x = train_x[j:(j+sequence_len),]
        seq_y = train_y[j:(j+sequence_len),]
        hiddens, outputs = forward(seq_x, layers)
        grad = mse_grad(seq_y, outputs)
        params = backward(layers, seq_x, lr, grad, hiddens)
        epoch_loss += mse(seq_y, outputs)

    if epoch % 50 == 0:
        sequence_len = 7
        valid_loss = 0
        for j in range(valid_x.shape[0] - sequence_len):
            seq_x = valid_x[j:(j+sequence_len),]
            seq_y = valid_y[j:(j+sequence_len),]
            _, outputs = forward(seq_x, layers)
            valid_loss += mse(seq_y, outputs)

        print(f"Epoch: {epoch} train loss {epoch_loss / len(train_x)} valid loss {valid_loss / len(valid_x)}")

Epoch: 0 train loss nan valid loss nan
